# Phase 6 (Abstractive Legal Summarization) - data prep
Pull gated HF data, clean, audit, analyse, tokenise with the Phase-1 tokenizer, build folds.

**Before running:** Internet ON, paste your HF token into `HF_TOKEN` in section 1, attach the Phase-1 tokenizer dataset.

**Output:** `/kaggle/working/summ_data_clean/`

## 1. Configuration

In [ ]:
import os
import re
import sys
import json
import glob
import time
import pickle
import shutil
import hashlib
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SPLITS = ["train", "dev", "test"]

# Paste your Hugging Face token here (hardcoded, no Kaggle secrets).
HF_TOKEN = "PASTE_YOUR_HF_TOKEN_HERE"

# Env vars are only for local testing. On Kaggle leave them alone.
LOCAL_DATA_DIR = os.environ.get("SUM_LOCAL_DATA_DIR", None)
TOKENIZER_DIR = os.environ.get("SUM_TOKENIZER_DIR", None)
OUT_DIR = os.environ.get("SUM_OUT_DIR", "/kaggle/working/summ_data_clean")

SEED = 42
N_FOLDS = 5

# Cleaning
KEEP_NEWLINES = True        # keep paragraph breaks in the document (summary is always one line)
EMPTY_PLACEHOLDER = "."     # used for an empty test document (a test case is never dropped)
MIN_DOC_WORDS = 20          # train/dev docs shorter than this are dropped
MIN_SUMMARY_WORDS = 3       # train/dev summaries shorter than this are dropped
DEV_FRACTION_IF_MISSING = 0.05   # only used if the repo has no dev file

# Leakage handling (same idea as Phase 4/5)
EXCLUDE_TEST_OVERLAP_FROM_POOL = True   # drop train/dev docs whose TEXT also appears in test

# Model context (tokens). Change if the Phase-1 models use another context length.
MODEL_CTX = 1024
SUMMARY_PERCENTILE = 98     # summary budget = this percentile of summary token lengths

# Long-document views
CHUNK_OVERLAP = 128
MAX_CHUNKS = 16
HEAD_FRAC = 0.5             # head share when keeping head + tail of a long document

# Chunk-aligned view (hierarchical summarization): each chunk gets its own target
CHUNK_DOC_LEN = 640         # tokens of document per chunk (target budget = MODEL_CTX - this - 2)
MIN_ALIGN_SCORE = 0.5       # a summary sentence counts as matched if this share of its words is in one doc sentence
FREQ_WORD_SKIP = 0.25       # ignore words found in more than this share of doc sentences when matching
ORACLE_SAMPLE = 300         # train docs used for the extractive-oracle check

# Analysis
ANALYSIS_SAMPLE = 2000      # docs used for the slower analyses
POSITION_SAMPLE = 300

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)
print("OUT_DIR:", OUT_DIR)

## 2. Hugging Face login

In [ ]:
if LOCAL_DATA_DIR is None:
    if HF_TOKEN is None or HF_TOKEN.startswith("PASTE_"):
        raise RuntimeError("Set HF_TOKEN in the configuration cell (section 1).")

    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
else:
    print("Local data mode, skipping Hugging Face login.")

## 3. Find and download the summarization files (train / dev / test)

In [ ]:
RAW_DIR = os.path.join(OUT_DIR, "raw_snapshot")
os.makedirs(RAW_DIR, exist_ok=True)

STEM_TO_SPLIT = {
    "train": "train",
    "dev": "dev",
    "val": "dev",
    "valid": "dev",
    "validation": "dev",
    "test": "test",
}


def simple_name(path):
    name = path.lower()
    name = name.replace("_", "")
    name = name.replace("-", "")
    name = name.replace(" ", "")
    return name


def is_summ_path(path):
    key = simple_name(path)
    if "summar" in key:
        return True
    if "phase6" in key:
        return True
    if "abstractive" in key:
        return True
    return False


candidates = {"train": [], "dev": [], "test": []}
all_files = []

if LOCAL_DATA_DIR is not None:
    for root, dirs, files in os.walk(LOCAL_DATA_DIR):
        for file_name in files:
            full_path = os.path.join(root, file_name)
            all_files.append(full_path)
            stem, ext = os.path.splitext(file_name.lower())
            if ext not in [".json", ".jsonl"]:
                continue
            if stem in STEM_TO_SPLIT and is_summ_path(full_path):
                candidates[STEM_TO_SPLIT[stem]].append(full_path)
else:
    from huggingface_hub import list_repo_files
    all_files = list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
    print("Files in repo:", len(all_files))
    for path in all_files:
        if not is_summ_path(path):
            continue
        stem, ext = os.path.splitext(os.path.basename(path).lower())
        if ext not in [".json", ".jsonl"]:
            continue
        if stem in STEM_TO_SPLIT:
            candidates[STEM_TO_SPLIT[stem]].append(path)

for split in SPLITS:
    print(split, "->", candidates[split])

if len(candidates["train"]) == 0:
    print("\nAll repo files, to help you spot the right ones:")
    for p in all_files:
        print("  ", p)
    raise FileNotFoundError("No train file found for the summarization phase.")

SPLIT_PATH = {}
for split in SPLITS:
    options = sorted(candidates[split], key=len)
    if len(options) == 0:
        print("NOTE: no file for split", split, "(will be handled below)")
        continue
    chosen = options[0]
    if len(options) > 1:
        print("WARNING: several candidates for", split, "- using the shortest path:", chosen)
    if LOCAL_DATA_DIR is not None:
        local_path = chosen
    else:
        from huggingface_hub import hf_hub_download
        local_path = hf_hub_download(HF_DATASET_REPO, chosen, repo_type="dataset", token=HF_TOKEN)
    ext = os.path.splitext(local_path)[1]
    saved_copy = os.path.join(RAW_DIR, split + ext)
    shutil.copy(local_path, saved_copy)
    SPLIT_PATH[split] = saved_copy
    print(split, "ready:", saved_copy, "|", round(os.path.getsize(saved_copy) / 1e6, 1), "MB")

## 4. Load (tolerant to several JSON layouts)

Each record: `id`, `document`, `summary` (the test set has no `summary`).

In [ ]:
FIELD_ALIASES = {
    "id": ["id", "doc_id", "document_id", "case_id", "idx"],
    "document": ["document", "text", "doc", "judgment", "body", "source", "article"],
    "summary": ["summary", "reference_summary", "abstract", "target", "headnote", "gold_summary"],
}


def pick_field(rec, kind):
    for name in FIELD_ALIASES[kind]:
        if name in rec:
            return rec[name]
    return None


def read_records(path):
    ext = os.path.splitext(path)[1].lower()
    with open(path, "r", encoding="utf-8") as f:
        if ext == ".jsonl":
            data = []
            for line in f:
                line = line.strip()
                if line:
                    data.append(json.loads(line))
        else:
            data = json.load(f)

    if isinstance(data, dict):
        for key in ["data", "documents", "examples", "records", "items"]:
            if key in data and isinstance(data[key], list):
                data = data[key]
                break

    if isinstance(data, dict):
        rows = []
        for key, value in data.items():
            if isinstance(value, dict):
                rec = dict(value)
                if "id" not in rec:
                    rec["id"] = key
            else:
                rec = {"id": key, "document": value}
            rows.append(rec)
        data = rows

    return data


raw = {}
for split in SPLITS:
    if split not in SPLIT_PATH:
        raw[split] = []
        continue
    raw[split] = read_records(SPLIT_PATH[split])
    print(split, "records:", len(raw[split]))

first = raw["train"][0]
print("keys in a train record:", list(first.keys()))
if pick_field(first, "document") is None:
    raise KeyError("Could not find a document field. Keys: " + str(list(first.keys())))
if pick_field(first, "summary") is None:
    raise KeyError("Could not find a summary field in train. Keys: " + str(list(first.keys())))
if len(raw["test"]) > 0:
    print("keys in a test record:", list(raw["test"][0].keys()))
    print("test has a summary field (should be False):", pick_field(raw["test"][0], "summary") is not None)
else:
    print("NOTE: no test records yet. Re-run this notebook when the test file is released.")

## 5. Cleaning

Document: unicode normalise, drop control chars, collapse spaces (paragraph breaks kept). Summary: one line.

In [ ]:
def clean_doc_text(text):
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    kept = []
    for ch in text:
        if ch == "\n" or ch == "\r":
            kept.append("\n")
        elif ch.isspace():
            kept.append(" ")
        elif unicodedata.category(ch) in ("Cc", "Cs"):
            continue
        else:
            kept.append(ch)
    text = "".join(kept)
    text = re.sub(r"[ ]+", " ", text)
    text = re.sub(r" ?\n ?", "\n", text)
    text = re.sub(r"\n{2,}", "\n", text)
    if not KEEP_NEWLINES:
        text = text.replace("\n", " ")
    return text.strip()


def clean_summary_text(text):
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    kept = []
    for ch in text:
        if ch.isspace():
            kept.append(" ")
        elif unicodedata.category(ch) in ("Cc", "Cs"):
            continue
        else:
            kept.append(ch)
    text = "".join(kept)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def text_hash(text):
    flat = re.sub(r"\s+", " ", text).strip().lower()
    return hashlib.md5(flat.encode("utf-8")).hexdigest()


def count_words(text):
    return len(text.split())


def clean_split(records, split):
    labeled = split in ["train", "dev"]
    rows = []
    stats = Counter()
    seen_ids = set()
    seen_pairs = set()
    for number, rec in enumerate(records):
        doc_id = pick_field(rec, "id")
        if doc_id is None:
            doc_id = "row_" + str(number)
            stats["missing_id_filled"] += 1
        doc_id = str(doc_id).strip()

        doc = clean_doc_text(pick_field(rec, "document"))
        summ = clean_summary_text(pick_field(rec, "summary")) if labeled else None

        if doc_id in seen_ids:
            stats["duplicate_id_dropped"] += 1
            continue

        if labeled:
            if doc == "":
                stats["empty_doc_dropped"] += 1
                continue
            if count_words(doc) < MIN_DOC_WORDS:
                stats["short_doc_dropped"] += 1
                continue
            if summ == "" or count_words(summ) < MIN_SUMMARY_WORDS:
                stats["short_summary_dropped"] += 1
                continue
            pair = text_hash(doc) + "|" + text_hash(summ)
            if pair in seen_pairs:
                stats["duplicate_pair_dropped"] += 1
                continue
            seen_pairs.add(pair)
        else:
            if doc == "":
                doc = EMPTY_PLACEHOLDER
                stats["empty_test_doc_placeholder"] += 1

        seen_ids.add(doc_id)
        rows.append({
            "id": doc_id,
            "document": doc,
            "summary": summ,
            "doc_words": count_words(doc),
            "sum_words": count_words(summ) if summ is not None else None,
            "doc_hash": text_hash(doc),
        })
    print(split, "raw:", len(records), "-> kept:", len(rows), "| dropped/notes:", dict(stats))
    return rows


clean_docs = {}
for split in SPLITS:
    clean_docs[split] = clean_split(raw[split], split)

# If the repo has no dev file, carve one out of train.
if len(clean_docs["dev"]) == 0:
    rng = np.random.RandomState(SEED)
    order = rng.permutation(len(clean_docs["train"]))
    n_dev = max(1, int(len(order) * DEV_FRACTION_IF_MISSING))
    dev_idx = set(order[:n_dev].tolist())
    new_train = []
    new_dev = []
    for i, row in enumerate(clean_docs["train"]):
        if i in dev_idx:
            new_dev.append(row)
        else:
            new_train.append(row)
    clean_docs["train"] = new_train
    clean_docs["dev"] = new_dev
    print("No dev file in repo: carved", len(new_dev), "docs out of train.")

## 6. Audit (leakage, duplicates, outliers)

In [ ]:
train_ids = set(r["id"] for r in clean_docs["train"])
dev_ids = set(r["id"] for r in clean_docs["dev"])
test_ids = set(r["id"] for r in clean_docs["test"])

print("id overlap train/dev :", len(train_ids & dev_ids))
print("id overlap train/test:", len(train_ids & test_ids))
print("id overlap dev/test  :", len(dev_ids & test_ids))

train_hash = set(r["doc_hash"] for r in clean_docs["train"])
dev_hash = set(r["doc_hash"] for r in clean_docs["dev"])
test_hash = set(r["doc_hash"] for r in clean_docs["test"])

print("same TEXT train/dev  :", len(train_hash & dev_hash))
print("same TEXT train/test :", len(train_hash & test_hash))
print("same TEXT dev/test   :", len(dev_hash & test_hash))

# Same id in two splits but different text = just a naming clash, not leakage.
for a, b in [("train", "test"), ("dev", "test")]:
    clash = 0
    b_by_id = {}
    for r in clean_docs[b]:
        b_by_id[r["id"]] = r["doc_hash"]
    for r in clean_docs[a]:
        if r["id"] in b_by_id and b_by_id[r["id"]] != r["doc_hash"]:
            clash += 1
    print("ids shared by", a, "and", b, "but with different text (naming clash only):", clash)

# Same document text in train with different summaries
by_hash = {}
for r in clean_docs["train"]:
    by_hash.setdefault(r["doc_hash"], []).append(r["id"])
multi = [v for v in by_hash.values() if len(v) > 1]
print("train docs whose text appears under several ids:", len(multi))

if EXCLUDE_TEST_OVERLAP_FROM_POOL and len(test_hash) > 0:
    for split in ["train", "dev"]:
        before = len(clean_docs[split])
        clean_docs[split] = [r for r in clean_docs[split] if r["doc_hash"] not in test_hash]
        print("removed from", split, "(same text as a test doc):", before - len(clean_docs[split]))
else:
    print("Test-overlap removal is off or there is no test set.")

# Outliers
all_labeled = clean_docs["train"] + clean_docs["dev"]
ratios = []
for r in all_labeled:
    ratios.append(r["sum_words"] / max(r["doc_words"], 1))
ratios = np.array(ratios)
print("\nsummary/doc length ratio: mean %.3f | p1 %.3f | p50 %.3f | p99 %.3f" % (
    ratios.mean(), np.percentile(ratios, 1), np.percentile(ratios, 50), np.percentile(ratios, 99)))

n_sum_longer = sum(1 for r in all_labeled if r["sum_words"] >= r["doc_words"])
n_copy = sum(1 for r in all_labeled if r["summary"] == r["document"])
print("summary at least as long as doc:", n_sum_longer, "| summary identical to doc:", n_copy)

non_ascii_fraction = []
for r in all_labeled[:ANALYSIS_SAMPLE]:
    text = r["document"]
    bad = sum(1 for ch in text if ord(ch) > 127)
    non_ascii_fraction.append(bad / max(len(text), 1))
print("mean non-ascii char fraction in docs (sample): %.4f" % float(np.mean(non_ascii_fraction)))

## 7. Length and compression analysis

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def describe(name, values):
    values = np.array(values)
    print("%-22s n=%d mean=%.0f p5=%.0f p50=%.0f p95=%.0f p99=%.0f max=%.0f" % (
        name, len(values), values.mean(), np.percentile(values, 5), np.percentile(values, 50),
        np.percentile(values, 95), np.percentile(values, 99), values.max()))

for split in SPLITS:
    if len(clean_docs[split]) > 0:
        describe(split + " doc words", [r["doc_words"] for r in clean_docs[split]])
for split in ["train", "dev"]:
    if len(clean_docs[split]) > 0:
        describe(split + " summary words", [r["sum_words"] for r in clean_docs[split]])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist([r["doc_words"] for r in clean_docs["train"]], bins=60)
axes[0].set_title("train doc words")
axes[1].hist([r["sum_words"] for r in clean_docs["train"]], bins=60)
axes[1].set_title("train summary words")
axes[2].scatter([r["doc_words"] for r in clean_docs["train"][:3000]],
                [r["sum_words"] for r in clean_docs["train"][:3000]], s=4)
axes[2].set_title("doc words vs summary words")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "length_plots.png"), dpi=110)
plt.show()

d_len = np.array([r["doc_words"] for r in clean_docs["train"]], dtype=float)
s_len = np.array([r["sum_words"] for r in clean_docs["train"]], dtype=float)
if len(d_len) > 2:
    print("correlation(doc words, summary words): %.3f" % np.corrcoef(d_len, s_len)[0, 1])

## 8. How extractive are the summaries? Where do they come from in the document?

Tells us which long-document strategy is worth trying (head, head+tail, chunks, selection).

In [ ]:
WORD_RE = re.compile(r"\w+")


def words_of(text):
    return WORD_RE.findall(text.lower())


def ngrams_of(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def unigram_f1(pred_words, ref_words):
    if len(pred_words) == 0 or len(ref_words) == 0:
        return 0.0
    pc = Counter(pred_words)
    rc = Counter(ref_words)
    overlap = 0
    for w, c in pc.items():
        if w in rc:
            overlap += min(c, rc[w])
    if overlap == 0:
        return 0.0
    p = overlap / len(pred_words)
    r = overlap / len(ref_words)
    return 2 * p * r / (p + r)


def split_sentences(text):
    pieces = []
    for para in text.split("\n"):
        for s in re.split(r"(?<=[.!?])\s+", para):
            s = s.strip()
            if len(s) > 0:
                pieces.append(s)
    return pieces


sample_rows = clean_docs["train"][:ANALYSIS_SAMPLE]

# Novel n-grams in the summary (how abstractive it is)
novel1 = []
novel2 = []
for r in sample_rows:
    dw = words_of(r["document"])
    sw = words_of(r["summary"])
    if len(sw) < 3:
        continue
    d1 = set(dw)
    d2 = set(ngrams_of(dw, 2))
    s1 = sw
    s2 = ngrams_of(sw, 2)
    novel1.append(sum(1 for w in s1 if w not in d1) / len(s1))
    novel2.append(sum(1 for g in s2 if g not in d2) / len(s2))
print("novel unigrams in summary: %.3f | novel bigrams: %.3f" % (np.mean(novel1), np.mean(novel2)))
print("(low = very extractive, high = very abstractive)")

# Lead-k and head+tail baselines, unigram F1 (quick ROUGE-1-like check)
dev_rows = clean_docs["dev"][:1000]
if len(dev_rows) == 0:
    dev_rows = clean_docs["train"][:1000]
print("\nlead-k baselines on", len(dev_rows), "docs (unigram F1, not official ROUGE):")
for k in [50, 100, 150, 200, 300, 400]:
    scores = []
    for r in dev_rows:
        dw = words_of(r["document"])
        scores.append(unigram_f1(dw[:k], words_of(r["summary"])))
    print("  first %3d words -> %.4f" % (k, np.mean(scores)))
for k in [200, 400]:
    scores = []
    for r in dev_rows:
        dw = words_of(r["document"])
        half = k // 2
        picked = dw[:half] + dw[-half:] if len(dw) > k else dw
        scores.append(unigram_f1(picked, words_of(r["summary"])))
    print("  head+tail %3d words -> %.4f" % (k, np.mean(scores)))

# Where in the document does each summary sentence come from?
bins = np.zeros(10)
total = 0
for r in sample_rows[:POSITION_SAMPLE]:
    doc_sents = split_sentences(r["document"])
    if len(doc_sents) < 10:
        continue
    doc_sets = [set(words_of(s)) for s in doc_sents]
    for ss in split_sentences(r["summary"]):
        sset = set(words_of(ss))
        if len(sset) < 4:
            continue
        best = -1
        best_score = 0.0
        for j, dset in enumerate(doc_sets):
            if len(dset) == 0:
                continue
            score = len(sset & dset) / len(sset)
            if score > best_score:
                best_score = score
                best = j
        if best >= 0 and best_score >= 0.5:
            rel = best / (len(doc_sents) - 1)
            b = min(int(rel * 10), 9)
            bins[b] += 1
            total += 1
if total > 0:
    print("\nrelative position of the document sentence that best matches each summary sentence:")
    for b in range(10):
        print("  %3d%%-%3d%%: %5.1f%%" % (b * 10, b * 10 + 10, 100.0 * bins[b] / total))
    plt.figure(figsize=(6, 3))
    plt.bar(range(10), bins / total)
    plt.title("source position of summary content")
    plt.savefig(os.path.join(OUT_DIR, "source_position.png"), dpi=110)
    plt.show()
else:
    print("Not enough matched sentences for the position analysis.")

## 9. Load the Phase-1 tokenizer

In [ ]:
from transformers import AutoTokenizer
os.environ["TOKENIZERS_PARALLELISM"] = "true"


def find_tokenizer_dir():
    if TOKENIZER_DIR is not None:
        return TOKENIZER_DIR
    paths = glob.glob("/kaggle/input/**/tokenizer.json", recursive=True)
    folders = []
    for p in paths:
        folder = os.path.dirname(p)
        if "checkpoint-" not in folder:
            folders.append(folder)
    folders = sorted(set(folders))
    if len(folders) == 0:
        raise FileNotFoundError("No tokenizer.json under /kaggle/input. Attach your tokenizer dataset (Add Input).")
    for folder in folders:
        if os.path.exists(os.path.join(folder, "trained_on.json")):
            return folder
    return folders[0]


tok_dir = find_tokenizer_dir()
tokenizer = AutoTokenizer.from_pretrained(tok_dir)
tokenizer.model_max_length = 10 ** 9
VOCAB_SIZE = len(tokenizer)
print("Tokenizer:", tok_dir, "| vocab size:", VOCAB_SIZE)


def find_sep_id(tok):
    if tok.eos_token_id is not None:
        return int(tok.eos_token_id)
    for name in ["<eos>", "</s>", "<|endoftext|>", "<EOS>"]:
        tid = tok.convert_tokens_to_ids(name)
        if tid is not None and tid != tok.unk_token_id:
            return int(tid)
    raise ValueError("Could not find an EOS-like token to use as separator.")


SEP_ID = find_sep_id(tokenizer)
if tokenizer.pad_token_id is not None:
    PAD_ID = int(tokenizer.pad_token_id)
else:
    PAD_ID = SEP_ID
print("SEP (EOS) id:", SEP_ID, "| PAD id:", PAD_ID)
if PAD_ID == SEP_ID:
    print("PAD and SEP share an id. In training mask by length / label -100, never by token id.")

if VOCAB_SIZE < 65535:
    ID_DTYPE = np.uint16
else:
    ID_DTYPE = np.int32
print("ids stored as:", ID_DTYPE.__name__)

ok = 0
sample_texts = [r["document"][:2000] for r in clean_docs["train"][:30]]
for t in sample_texts:
    ids = tokenizer(t, add_special_tokens=False)["input_ids"]
    back = tokenizer.decode(ids, clean_up_tokenization_spaces=False)
    if back == t:
        ok += 1
print("round-trip exact match:", ok, "/", len(sample_texts))
if ok < len(sample_texts):
    print("Not exact: the tokenizer normalises or lowercases text. Fine as long as it is the same one Phase 1 used.")

## 10. Tokenise everything and set the token budgets

In [ ]:
def tokenize_texts(texts, batch_size=64):
    out = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        enc = tokenizer(batch, add_special_tokens=False)["input_ids"]
        for ids in enc:
            out.append(np.array(ids, dtype=ID_DTYPE))
        if (start // batch_size) % 50 == 0:
            print("  tokenised", min(start + batch_size, len(texts)), "/", len(texts))
    return out


t0 = time.time()
for split in SPLITS:
    rows = clean_docs[split]
    if len(rows) == 0:
        continue
    print("tokenising", split, "documents")
    doc_ids_list = tokenize_texts([r["document"] for r in rows])
    for r, ids in zip(rows, doc_ids_list):
        r["doc_ids"] = ids
        r["doc_tokens"] = int(len(ids))
    if split != "test":
        print("tokenising", split, "summaries")
        sum_ids_list = tokenize_texts([r["summary"] for r in rows])
        for r, ids in zip(rows, sum_ids_list):
            r["sum_ids"] = ids
            r["sum_tokens"] = int(len(ids))
print("tokenisation took %.1f s" % (time.time() - t0))

describe("train doc tokens", [r["doc_tokens"] for r in clean_docs["train"]])
describe("train summary tokens", [r["sum_tokens"] for r in clean_docs["train"]])
if len(clean_docs["test"]) > 0:
    describe("test doc tokens", [r["doc_tokens"] for r in clean_docs["test"]])

tpw = np.mean([r["doc_tokens"] / max(r["doc_words"], 1) for r in clean_docs["train"][:2000]])
print("tokens per word (docs): %.2f" % tpw)

# Budgets: [doc] SEP [summary] SEP must fit in MODEL_CTX
sum_tok = np.array([r["sum_tokens"] for r in clean_docs["train"]])
raw_budget = int(np.percentile(sum_tok, SUMMARY_PERCENTILE))
SUMMARY_BUDGET = int(np.ceil(raw_budget / 32.0) * 32)
SUMMARY_BUDGET = min(SUMMARY_BUDGET, MODEL_CTX // 2)
DOC_BUDGET = MODEL_CTX - SUMMARY_BUDGET - 2
print("\nsummary budget (p%d, rounded): %d tokens" % (SUMMARY_PERCENTILE, SUMMARY_BUDGET))
print("document budget: %d tokens (context %d)" % (DOC_BUDGET, MODEL_CTX))
print("summaries longer than budget (cut in training):",
      int((sum_tok > SUMMARY_BUDGET).sum()), "of", len(sum_tok))

# How much of each document survives each strategy?
doc_tok = np.array([r["doc_tokens"] for r in clean_docs["train"]])
fits = (doc_tok <= DOC_BUDGET).mean()
kept_head = np.minimum(doc_tok, DOC_BUDGET) / doc_tok
print("\ndocs that fit in one context: %.1f%%" % (100 * fits))
print("share of doc tokens kept by head-only truncation: mean %.3f | median %.3f" % (
    kept_head.mean(), np.median(kept_head)))

n_chunks = np.ceil(np.maximum(doc_tok - CHUNK_OVERLAP, 1) / float(DOC_BUDGET - CHUNK_OVERLAP))
n_chunks = np.maximum(n_chunks, 1)
print("chunks per doc (len %d, overlap %d): mean %.1f | p50 %.0f | p95 %.0f | max %.0f" % (
    DOC_BUDGET, CHUNK_OVERLAP, n_chunks.mean(), np.median(n_chunks), np.percentile(n_chunks, 95), n_chunks.max()))
print("docs needing more than MAX_CHUNKS (%d): %.1f%%" % (MAX_CHUNKS, 100 * (n_chunks > MAX_CHUNKS).mean()))

plt.figure(figsize=(6, 3))
plt.hist(np.minimum(doc_tok, 20000), bins=60)
plt.axvline(DOC_BUDGET, color="red")
plt.title("train doc tokens (red = doc budget)")
plt.savefig(os.path.join(OUT_DIR, "doc_tokens.png"), dpi=110)
plt.show()

## 11. Chunk-aligned view (for hierarchical summarization)

1. Split each document into sentences and cut it into chunks at sentence boundaries (about `CHUNK_DOC_LEN` tokens).
2. Match every summary sentence to the document sentence it came from.
3. Each chunk's target = the summary sentences that came from that chunk (empty target = nothing to say).

At inference the model summarizes every chunk and the pieces are joined in order.

In [ ]:
import bisect

CHUNK_SUM_BUDGET = MODEL_CTX - CHUNK_DOC_LEN - 2
print("chunk view: doc %d + target %d + 2 SEP = ctx %d" % (CHUNK_DOC_LEN, CHUNK_SUM_BUDGET, MODEL_CTX))


def sentence_spans(text):
    spans = []
    start = 0
    for m in re.finditer(r"(?<=[.!?])[ \t]+|\n+", text):
        end = m.start()
        if end > start:
            spans.append((start, end))
        start = m.end()
    if start < len(text):
        spans.append((start, len(text)))
    return spans


def make_bounds(sent_tok_starts, n_tokens, max_len):
    # chunk boundaries (token positions), cut at sentence starts where possible
    if n_tokens == 0:
        return [0, 0]
    cuts = sorted(set(c for c in sent_tok_starts if 0 < c < n_tokens))
    bounds = [0]
    start = 0
    while n_tokens - start > max_len:
        limit = start + max_len
        idx = bisect.bisect_right(cuts, limit) - 1
        if idx >= 0 and cuts[idx] > start:
            cut = cuts[idx]
        else:
            cut = limit
        bounds.append(cut)
        start = cut
    bounds.append(n_tokens)
    return bounds


def align_summary_to_doc(doc_text, doc_spans, sum_sents):
    # for every summary sentence: (best doc sentence index, share of its words found there)
    postings = {}
    for j, (a, b) in enumerate(doc_spans):
        for w in set(words_of(doc_text[a:b])):
            postings.setdefault(w, []).append(j)
    n = len(doc_spans)
    skip_limit = max(20, int(FREQ_WORD_SKIP * n))
    results = []
    for sent in sum_sents:
        sw = set(words_of(sent))
        usable = [w for w in sw if (w not in postings) or len(postings[w]) <= skip_limit]
        in_doc = [w for w in usable if w in postings]
        if len(usable) == 0 or len(in_doc) == 0:
            results.append((-1, 0.0))
            continue
        counts = {}
        for w in in_doc:
            for j in postings[w]:
                counts[j] = counts.get(j, 0) + 1
        best_j = max(counts, key=counts.get)
        results.append((best_j, counts[best_j] / len(usable)))
    return results


def assign_chunks(matches, chunk_of_sent, n_chunks):
    m = len(matches)
    assigned = [-1] * m
    for i, (j, score) in enumerate(matches):
        if j >= 0 and score >= MIN_ALIGN_SCORE:
            assigned[i] = chunk_of_sent[j]
    n_matched = sum(1 for a in assigned if a >= 0)
    if n_matched == 0:
        for i in range(m):
            assigned[i] = min(int(i / max(m, 1) * n_chunks), n_chunks - 1)
        return assigned, 0
    last = -1
    for i in range(m):
        if assigned[i] >= 0:
            last = assigned[i]
        elif last >= 0:
            assigned[i] = last
    first = -1
    for a in assigned:
        if a >= 0:
            first = a
            break
    for i in range(m):
        if assigned[i] < 0:
            assigned[i] = first
        else:
            break
    return assigned, n_matched


def build_aligned(row, labeled, want_oracle, check_ids):
    text = row["document"]
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = enc["input_ids"]
    offs = enc["offset_mapping"]
    n_tok = len(ids)
    if check_ids:
        assert np.array_equal(np.array(ids, dtype=ID_DTYPE), row["doc_ids"]), "token ids differ from section 10"

    ends = np.maximum.accumulate(np.array([o[1] for o in offs], dtype=np.int64)) if n_tok > 0 else np.zeros(0, dtype=np.int64)
    spans = sentence_spans(text)
    sent_tok_start = [int(np.searchsorted(ends, a, side="right")) for a, b in spans]
    bounds = make_bounds(sent_tok_start, n_tok, CHUNK_DOC_LEN)
    n_chunks = len(bounds) - 1

    entry = {"bounds": np.array(bounds, dtype=np.int32), "targets": None,
             "n_sum_sent": 0, "n_matched": 0, "mean_score": 0.0}
    oracle = None
    if not labeled:
        return entry, oracle

    chunk_of_sent = []
    for s in sent_tok_start:
        k = bisect.bisect_right(bounds, s) - 1
        chunk_of_sent.append(min(max(k, 0), n_chunks - 1))

    sum_spans = sentence_spans(row["summary"])
    sum_sents = [row["summary"][a:b] for a, b in sum_spans]
    matches = align_summary_to_doc(text, spans, sum_sents)
    assigned, n_matched = assign_chunks(matches, chunk_of_sent, n_chunks)

    texts = []
    for k in range(n_chunks):
        pieces = [sum_sents[i] for i in range(len(sum_sents)) if assigned[i] == k]
        texts.append(" ".join(pieces))
    enc_t = tokenizer(texts, add_special_tokens=False)["input_ids"]
    entry["targets"] = [np.array(t, dtype=ID_DTYPE) for t in enc_t]
    entry["n_sum_sent"] = len(sum_sents)
    entry["n_matched"] = n_matched
    entry["mean_score"] = float(np.mean([s for j, s in matches])) if len(matches) > 0 else 0.0

    if want_oracle:
        picked = sorted(set(j for j, s in matches if j >= 0 and s >= MIN_ALIGN_SCORE))
        oracle_text = " ".join(text[spans[j][0]:spans[j][1]] for j in picked)
        oracle = unigram_f1(words_of(oracle_text), words_of(row["summary"]))
    return entry, oracle


aligned = {"train": {}, "dev": {}, "test": {}}
oracle_scores = []
chunks_per_doc = []
target_tokens = []
empty_targets = 0
total_chunks = 0
sent_total = 0
sent_matched = 0

t0 = time.time()
for split in SPLITS:
    rows = clean_docs[split]
    labeled = split != "test"
    for number, row in enumerate(rows):
        want_oracle = (split == "train") and (number < ORACLE_SAMPLE)
        entry, oracle = build_aligned(row, labeled, want_oracle, number < 20)
        aligned[split][row["id"]] = entry
        chunks_per_doc.append(len(entry["bounds"]) - 1)
        if oracle is not None:
            oracle_scores.append(oracle)
        if labeled:
            for t in entry["targets"]:
                target_tokens.append(len(t))
                total_chunks += 1
                if len(t) == 0:
                    empty_targets += 1
            sent_total += entry["n_sum_sent"]
            sent_matched += entry["n_matched"]
        if (number + 1) % 1000 == 0:
            print("  aligned", split, number + 1, "/", len(rows), "| %.0f s" % (time.time() - t0))
print("alignment took %.1f s" % (time.time() - t0))

describe("chunks per doc", chunks_per_doc)
target_tokens = np.array(target_tokens)
describe("target tokens per chunk", target_tokens)
print("empty chunk targets: %.1f%%" % (100.0 * empty_targets / max(total_chunks, 1)))
print("targets longer than budget (%d): %.1f%%" % (CHUNK_SUM_BUDGET, 100.0 * (target_tokens > CHUNK_SUM_BUDGET).mean()))
print("summary sentences matched (score >= %.2f): %.1f%%" % (MIN_ALIGN_SCORE, 100.0 * sent_matched / max(sent_total, 1)))
if len(oracle_scores) > 0:
    print("extractive oracle (matched doc sentences vs reference), unigram F1: %.4f" % np.mean(oracle_scores))
    print("(upper-ish bound for a purely extractive method; compare with the lead-k numbers above)")

sum_ratio = np.median([r["sum_tokens"] / max(r["doc_tokens"], 1) for r in clean_docs["train"]])
print("median summary/doc token ratio: %.3f (use it for generation length control)" % sum_ratio)

plt.figure(figsize=(6, 3))
plt.hist(np.minimum(target_tokens, 1000), bins=60)
plt.axvline(CHUNK_SUM_BUDGET, color="red")
plt.title("chunk target tokens (red = budget)")
plt.savefig(os.path.join(OUT_DIR, "chunk_targets.png"), dpi=110)
plt.show()

## 12. Folds (stratified by document length)

In [ ]:
from sklearn.model_selection import StratifiedKFold

pool = clean_docs["train"] + clean_docs["dev"]
pool_tokens = np.array([r["doc_tokens"] for r in pool], dtype=float)
edges = np.percentile(pool_tokens, [20, 40, 60, 80])
length_bin = np.digitize(pool_tokens, edges)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_of = np.zeros(len(pool), dtype=int)
for fold_number, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(pool)), length_bin)):
    fold_of[va_idx] = fold_number
for r, f in zip(pool, fold_of):
    r["fold"] = int(f)
for r in clean_docs["test"]:
    r["fold"] = -1

print("pool (train + dev):", len(pool))
print("fold sizes:", Counter(fold_of.tolist()))
print("train docs per fold:", Counter(r["fold"] for r in clean_docs["train"]))
print("dev docs per fold:", Counter(r["fold"] for r in clean_docs["dev"]))

## 13. Save the bundle and the helper file

In [ ]:
bundle = {
    "train": clean_docs["train"],
    "dev": clean_docs["dev"],
    "test": clean_docs["test"],
    "meta": {
        "vocab_size": VOCAB_SIZE,
        "sep_id": SEP_ID,
        "pad_id": PAD_ID,
        "id_dtype": ID_DTYPE.__name__,
        "model_ctx": MODEL_CTX,
        "summary_budget": SUMMARY_BUDGET,
        "doc_budget": DOC_BUDGET,
        "chunk_overlap": CHUNK_OVERLAP,
        "max_chunks": MAX_CHUNKS,
        "head_frac": HEAD_FRAC,
        "chunk_doc_len": CHUNK_DOC_LEN,
        "chunk_sum_budget": CHUNK_SUM_BUDGET,
        "min_align_score": MIN_ALIGN_SCORE,
        "median_sum_ratio": float(sum_ratio),
        "n_folds": N_FOLDS,
        "seed": SEED,
        "tokenizer_dir": tok_dir,
        "n_train": len(clean_docs["train"]),
        "n_dev": len(clean_docs["dev"]),
        "n_test": len(clean_docs["test"]),
    },
}

pkl_path = os.path.join(OUT_DIR, "tokenized_summ.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)
print("saved:", pkl_path, "|", round(os.path.getsize(pkl_path) / 1e6, 1), "MB")

aligned["meta"] = bundle["meta"]
aligned_path = os.path.join(OUT_DIR, "aligned_summ.pkl")
with open(aligned_path, "wb") as f:
    pickle.dump(aligned, f, protocol=pickle.HIGHEST_PROTOCOL)
print("saved:", aligned_path, "|", round(os.path.getsize(aligned_path) / 1e6, 1), "MB")

# Plain cleaned text (no token ids), handy for ROUGE / BERTScore later
for split in SPLITS:
    rows = []
    for r in clean_docs[split]:
        item = {"id": r["id"], "document": r["document"]}
        if r["summary"] is not None:
            item["summary"] = r["summary"]
        item["fold"] = r["fold"]
        rows.append(item)
    with open(os.path.join(OUT_DIR, "clean_" + split + ".json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False)

stat_rows = []
for split in SPLITS:
    for r in clean_docs[split]:
        stat_rows.append({
            "split": split, "id": r["id"], "fold": r["fold"],
            "doc_words": r["doc_words"], "sum_words": r["sum_words"],
            "doc_tokens": r["doc_tokens"], "sum_tokens": r.get("sum_tokens"),
        })
pd.DataFrame(stat_rows).to_csv(os.path.join(OUT_DIR, "length_stats.csv"), index=False)

HELPER_SOURCE = r"""
import re
import json
import pickle
from collections import Counter

import numpy as np


def load_bundle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def head_tail_ids(ids, budget, head_frac=0.5):
    # keep the start and the end of a long document
    if len(ids) <= budget:
        return np.asarray(ids)
    head_len = int(budget * head_frac)
    tail_len = budget - head_len
    head = ids[:head_len]
    tail = ids[len(ids) - tail_len:]
    return np.concatenate([head, tail])


def head_ids(ids, budget):
    return np.asarray(ids[:budget])


def make_chunks(ids, chunk_len, overlap, max_chunks):
    # overlapping chunks; if too many, keep evenly spaced ones (first and last always kept)
    n = len(ids)
    if n <= chunk_len:
        return [np.asarray(ids)]
    step = chunk_len - overlap
    starts = list(range(0, n - overlap, step))
    chunks = []
    for s in starts:
        chunks.append(np.asarray(ids[s:s + chunk_len]))
    if len(chunks) > max_chunks:
        keep = np.linspace(0, len(chunks) - 1, max_chunks).round().astype(int)
        chunks = [chunks[i] for i in sorted(set(keep.tolist()))]
    return chunks


def build_lm_example(doc_ids, sum_ids, sep_id, doc_budget, summary_budget, strategy="head", head_frac=0.5):
    # input: [doc] SEP [summary] SEP ; labels are -100 on the doc part, so loss is on the summary only
    if strategy == "head_tail":
        d = head_tail_ids(doc_ids, doc_budget, head_frac)
    else:
        d = head_ids(doc_ids, doc_budget)
    s = np.asarray(sum_ids[:summary_budget])
    input_ids = np.concatenate([d, [sep_id], s, [sep_id]]).astype(np.int64)
    labels = np.full(len(input_ids), -100, dtype=np.int64)
    start = len(d) + 1
    labels[start:] = input_ids[start:]
    return input_ids, labels


def build_prompt(doc_ids, sep_id, doc_budget, strategy="head", head_frac=0.5):
    # inference prompt: [doc] SEP  (the model then writes the summary until SEP)
    if strategy == "head_tail":
        d = head_tail_ids(doc_ids, doc_budget, head_frac)
    else:
        d = head_ids(doc_ids, doc_budget)
    return np.concatenate([d, [sep_id]]).astype(np.int64)


WORD_RE = re.compile(r"\w+")


def _words(text):
    return WORD_RE.findall(text.lower())


def load_aligned(path):
    # dict: split -> {doc id -> {"bounds", "targets", ...}} plus "meta"
    with open(path, "rb") as f:
        return pickle.load(f)


def chunk_slices(ids, bounds):
    # the document cut at the stored chunk boundaries (exact token ids, nothing lost)
    out = []
    for k in range(len(bounds) - 1):
        out.append(np.asarray(ids[bounds[k]:bounds[k + 1]]))
    return out


def build_chunk_example(chunk_ids, target_ids, sep_id, chunk_budget, target_budget):
    # [chunk] SEP [chunk target] SEP ; an empty target means the model should write only SEP
    return build_lm_example(chunk_ids, target_ids, sep_id, chunk_budget, target_budget)


def build_chunk_prompt(chunk_ids, sep_id, chunk_budget):
    return build_prompt(chunk_ids, sep_id, chunk_budget)


def join_chunk_summaries(texts):
    # final document summary = chunk summaries in order, empty ones skipped
    pieces = []
    for t in texts:
        t = t.strip()
        if t != "":
            pieces.append(t)
    return " ".join(pieces)


def rouge1_f1(pred, ref):
    pw = _words(pred)
    rw = _words(ref)
    if len(pw) == 0 or len(rw) == 0:
        return 0.0
    pc = Counter(pw)
    rc = Counter(rw)
    overlap = 0
    for w, c in pc.items():
        if w in rc:
            overlap += min(c, rc[w])
    if overlap == 0:
        return 0.0
    p = overlap / len(pw)
    r = overlap / len(rw)
    return 2 * p * r / (p + r)


def rougeL_f1(pred, ref):
    pw = _words(pred)
    rw = _words(ref)
    if len(pw) == 0 or len(rw) == 0:
        return 0.0
    prev = [0] * (len(rw) + 1)
    for i in range(1, len(pw) + 1):
        cur = [0] * (len(rw) + 1)
        for j in range(1, len(rw) + 1):
            if pw[i - 1] == rw[j - 1]:
                cur[j] = prev[j - 1] + 1
            else:
                cur[j] = max(prev[j], cur[j - 1])
        prev = cur
    lcs = prev[len(rw)]
    if lcs == 0:
        return 0.0
    p = lcs / len(pw)
    r = lcs / len(rw)
    return 2 * p * r / (p + r)


def repetition_rate(text, n=3):
    # share of repeated word n-grams in a generated summary (0 = no repetition)
    w = _words(text)
    if len(w) < n + 1:
        return 0.0
    grams = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return 1.0 - len(set(grams)) / len(grams)


def write_submission(test_rows, summaries, path, fallback_words=60):
    # test_rows: bundle["test"]; summaries: dict id -> text. One summary per test id, list format.
    out = []
    for r in test_rows:
        text = summaries.get(r["id"], "")
        if text is None or text.strip() == "":
            text = " ".join(r["document"].split()[:fallback_words])
        out.append({"id": r["id"], "summary": text.strip()})
    ids = [o["id"] for o in out]
    assert len(ids) == len(set(ids)), "duplicate ids in submission"
    assert len(out) == len(test_rows), "wrong number of summaries"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=1)
    print("wrote", len(out), "summaries to", path)
    return out
"""

helper_path = os.path.join(OUT_DIR, "summ_utils.py")
with open(helper_path, "w", encoding="utf-8") as f:
    f.write(HELPER_SOURCE.strip() + "\n")
print("saved:", helper_path)

# Quick self-test of the helpers on real data
sys.path.insert(0, OUT_DIR)
import importlib
import summ_utils
importlib.reload(summ_utils)

ex = clean_docs["train"][0]
inp, lab = summ_utils.build_lm_example(ex["doc_ids"], ex["sum_ids"], SEP_ID, DOC_BUDGET, SUMMARY_BUDGET)
print("example length:", len(inp), "<= ctx", MODEL_CTX, "| supervised tokens:", int((lab != -100).sum()))
assert len(inp) <= MODEL_CTX
chunks = summ_utils.make_chunks(ex["doc_ids"], DOC_BUDGET, CHUNK_OVERLAP, MAX_CHUNKS)
print("chunks for first doc:", len(chunks), "| lengths:", [len(c) for c in chunks][:5])
print("rouge1 of summary vs itself:", round(summ_utils.rouge1_f1(ex["summary"], ex["summary"]), 3))

entry = aligned["train"][ex["id"]]
pieces = summ_utils.chunk_slices(ex["doc_ids"], entry["bounds"])
assert np.array_equal(np.concatenate(pieces), np.asarray(ex["doc_ids"])), "chunks do not rebuild the document"
for k in range(len(pieces)):
    c_inp, c_lab = summ_utils.build_chunk_example(pieces[k], entry["targets"][k], SEP_ID, CHUNK_DOC_LEN, CHUNK_SUM_BUDGET)
    assert len(c_inp) <= MODEL_CTX
print("chunk view ok:", len(pieces), "chunks | first chunk example length:", len(summ_utils.build_chunk_example(pieces[0], entry["targets"][0], SEP_ID, CHUNK_DOC_LEN, CHUNK_SUM_BUDGET)[0]))

## 14. Summary

In [ ]:
print("train docs:", len(clean_docs["train"]))
print("dev docs  :", len(clean_docs["dev"]))
print("test docs :", len(clean_docs["test"]))
print("context %d = doc %d + summary %d + 2 SEP" % (MODEL_CTX, DOC_BUDGET, SUMMARY_BUDGET))
print("\nfiles in", OUT_DIR)
for name in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, name)
    if os.path.isfile(full):
        print("  ", name, round(os.path.getsize(full) / 1e6, 2), "MB")
print("\nNext: attach this output to the training notebooks (they auto-find tokenized_summ.pkl + summ_utils.py).")